# 01 — Bronze Layer: Data Loading

Bu notebook Olist layihəsi üçün ilkin məlumatların Bronze qatına yüklənməsini və yüklənmiş cədvəllərin yoxlanmasını təşkil edir.

**Axın:** kitabxanalar → verilənlər bazasının hazırlanması → CSV faylları → xarici API məlumatları → nəticələrin yoxlanması.

## 1. Import libraries

Notebook boyunca istifadə olunan Python kitabxanaları və database bağlantısı.

In [ ]:
import time
from datetime import date, timedelta
from pathlib import Path
import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError
from sqlalchemy.engine import URL
from db import get_engine
import os
import json


## 2. Initialize the database

Database və tələb olunan sxemlərin ilkin hazırlanması.

In [ ]:
from db import setup_database
setup_database()


## 3. Load local CSV files into Bronze

Dataset qovluğundakı CSV faylları oxunur və `bronze` sxeminə yüklənir. Bağlantı xətası olduqda kod müəyyən sayda yenidən cəhd edir.

In [ ]:
DATA_DIR = Path("/workspaces/olist-ecommerce-analytics/data/olist_dataset")
MAX_ATTEMPTS = 5
RETRY_DELAY_SECONDS = 15

for f in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(f, dtype=str, keep_default_na=False)

    for attempt in range(1, MAX_ATTEMPTS + 1):
        engine = get_engine()
        try:
            df.to_sql(f.stem, engine, schema="bronze",
                      if_exists="replace", index=False, chunksize=1000)
        except OperationalError as exc:
            if attempt == MAX_ATTEMPTS:
                raise RuntimeError(
                    f"SQL Server did not recover; loading {f.name} failed "
                    f"after {MAX_ATTEMPTS} attempts."
                ) from exc
            print(
                f"Database connection failed while loading {f.name}; "
                f"retrying in {RETRY_DELAY_SECONDS}s "
                f"({attempt}/{MAX_ATTEMPTS})..."
            )
            retry = True
        else:
            retry = False
        finally:
            engine.dispose()

        if not retry:
            print(f"{f.stem}: {len(df):,} sətir yükləndi")
            break
        time.sleep(RETRY_DELAY_SECONDS)

## 4. Prepare API data loading

API-dən gələn məlumatların JSON faylları üçün qovluq yaradılır və database engine alınır.

In [ ]:


# 1. 'data/api/' qovluğunu yaradırıq
api_dir = Path("../data/api")
api_dir.mkdir(parents=True, exist_ok=True)

engine = get_engine()



## 5. Load public holidays — Nager.Date API

Braziliyanın 2016–2018-ci illər üzrə bayram məlumatları alınır, JSON faylına yazılır və `bronze.api_holidays` cədvəlinə yüklənir.

In [ ]:
# =============================================================
# 1. Nager.Date API - Braziliya Tətilləri (Holidays)
# =============================================================
print("1. Bayram məlumatları çəkilir (Nager.Date)...")
holidays_list = []
for year in [2016, 2017, 2018]:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/BR"
    res = requests.get(url)
    if res.status_code == 200:
        holidays_list.extend(res.json())

# JSON olaraq save edirik
holidays_json_path = api_dir / "br_holidays.json"
with open(holidays_json_path, "w", encoding="utf-8") as f:
    json.dump(holidays_list, f, ensure_ascii=False, indent=2)

# DataFrame-ə çeviririk və list/dict tiplərini string-ə çeviririk (SQL xətası olmasın)
df_holidays = pd.DataFrame(holidays_list)
for col in df_holidays.columns:
    if df_holidays[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df_holidays[col] = df_holidays[col].apply(lambda x: json.dumps(x) if x is not None else None)

# SQL Server Bronze-a yükləyirik
df_holidays.to_sql("api_holidays", con=engine, schema="bronze", if_exists="replace", index=False)
print(f"-> {len(df_holidays)} sətir bayram məlumatı 'bronze.api_holidays' cədvəlinə yazıldı.")



## 6. Load exchange rates — Frankfurter API

2016–2018-ci illər üçün BRL → USD məzənnələri alınır, JSON faylına yazılır və `bronze.api_exchange_rates` cədvəlinə yüklənir.

In [ ]:
# =============================================================
# 2. Frankfurter API - Valyuta Məzənnəsi (BRL -> USD)
# =============================================================
print("\n2. Valyuta kursları çəkilir (Frankfurter)...")
url_forex = "https://api.frankfurter.app/2016-01-01..2018-12-31?from=BRL&to=USD"
res_forex = requests.get(url_forex)
forex_data = res_forex.json()

# JSON olaraq save edirik
forex_json_path = api_dir / "brl_usd_exchange_rate.json"
with open(forex_json_path, "w", encoding="utf-8") as f:
    json.dump(forex_data, f, ensure_ascii=False, indent=2)

rates_dict = forex_data.get("rates", {})
df_forex = pd.DataFrame([
    {"date": date, "rate_usd": rates["USD"]} 
    for date, rates in rates_dict.items()
])

# SQL Server Bronze-a yükləyirik
df_forex.to_sql("api_exchange_rates", con=engine, schema="bronze", if_exists="replace", index=False)
print(f"-> {len(df_forex)} sətir valyuta məlumatı 'bronze.api_exchange_rates' cədvəlinə yazıldı.")



## 7. Load weather data — Open-Meteo API

São Paulo üçün gündəlik hava məlumatları alınır, JSON faylına yazılır və `bronze.api_weather` cədvəlinə yüklənir.

In [ ]:
# =============================================================
# 3. Open-Meteo API - Hava Şəraiti (Weather - San Paulo)
# =============================================================
print("\n3. Hava məlumatları çəkilir (Open-Meteo)...")
lat, lon = -23.5505, -46.6333
url_weather = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={lat}&longitude={lon}&start_date=2016-01-01&end_date=2018-12-31&"
    f"daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America%2FSao_Paulo"
)

res_weather = requests.get(url_weather)
weather_data = res_weather.json()

# JSON olaraq save edirik
weather_json_path = api_dir / "sao_paulo_weather.json"
with open(weather_json_path, "w", encoding="utf-8") as f:
    json.dump(weather_data, f, ensure_ascii=False, indent=2)

daily = weather_data.get("daily", {})
df_weather = pd.DataFrame({
    "date": daily.get("time", []),
    "temp_max": daily.get("temperature_2m_max", []),
    "temp_min": daily.get("temperature_2m_min", []),
    "precipitation": daily.get("precipitation_sum", [])
})

# SQL Server Bronze-a yükləyirik
df_weather.to_sql("api_weather", con=engine, schema="bronze", if_exists="replace", index=False)
print(f"-> {len(df_weather)} sətir hava məlumatı 'bronze.api_weather' cədvəlinə yazıldı.")

print("\n BÜTÜN API MƏLUMATLARI BRONZE QATINA XƏTASIZ YÜKLƏNDİ!")

## 8. Verify Bronze tables

Bronze sxemindəki cədvəllərin adları və sətir sayları yoxlanılır.

In [ ]:
with get_engine().connect() as conn:
    print(pd.read_sql(text("""
        SELECT s.name AS schema_adi,
               t.name AS cedvel,
               SUM(p.rows) AS setir_sayi
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0,1)
        WHERE s.name = 'bronze'
        GROUP BY s.name, t.name
        ORDER BY t.name
    """), conn))

In [ ]:
from db import setup_database
setup_database()


Microsoft SQL Server 2022 (RTM-CU27-GDR) (KB5126934) - 16.0.
Logged in as: datacube_analytics
     name
0  bronze
1    gold
2  silver


In [5]:
DATA_DIR = Path("/workspaces/olist-ecommerce-analytics/data/olist_dataset")
MAX_ATTEMPTS = 5
RETRY_DELAY_SECONDS = 15

for f in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(f, dtype=str, keep_default_na=False)

    for attempt in range(1, MAX_ATTEMPTS + 1):
        engine = get_engine()
        try:
            df.to_sql(f.stem, engine, schema="bronze",
                      if_exists="replace", index=False, chunksize=1000)
        except OperationalError as exc:
            if attempt == MAX_ATTEMPTS:
                raise RuntimeError(
                    f"SQL Server did not recover; loading {f.name} failed "
                    f"after {MAX_ATTEMPTS} attempts."
                ) from exc
            print(
                f"Database connection failed while loading {f.name}; "
                f"retrying in {RETRY_DELAY_SECONDS}s "
                f"({attempt}/{MAX_ATTEMPTS})..."
            )
            retry = True
        else:
            retry = False
        finally:
            engine.dispose()

        if not retry:
            print(f"{f.stem}: {len(df):,} sətir yükləndi")
            break
        time.sleep(RETRY_DELAY_SECONDS)

olist_customers_dataset: 99,441 sətir yükləndi
olist_geolocation_dataset: 1,000,163 sətir yükləndi
olist_order_items_dataset: 112,650 sətir yükləndi
olist_order_payments_dataset: 103,886 sətir yükləndi
olist_order_reviews_dataset: 99,224 sətir yükləndi
olist_orders_dataset: 99,441 sətir yükləndi
olist_products_dataset: 32,951 sətir yükləndi
olist_sellers_dataset: 3,095 sətir yükləndi
product_category_name_translation: 71 sətir yükləndi


In [5]:


# 1. 'data/api/' qovluğunu yaradırıq
api_dir = Path("../data/api")
api_dir.mkdir(parents=True, exist_ok=True)

engine = get_engine()

# =============================================================
# 1. Nager.Date API - Braziliya Tətilləri (Holidays)
# =============================================================
print("1. Bayram məlumatları çəkilir (Nager.Date)...")
holidays_list = []
for year in [2016, 2017, 2018]:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/BR"
    res = requests.get(url)
    if res.status_code == 200:
        holidays_list.extend(res.json())

# JSON olaraq save edirik
holidays_json_path = api_dir / "br_holidays.json"
with open(holidays_json_path, "w", encoding="utf-8") as f:
    json.dump(holidays_list, f, ensure_ascii=False, indent=2)

# DataFrame-ə çeviririk və list/dict tiplərini string-ə çeviririk (SQL xətası olmasın)
df_holidays = pd.DataFrame(holidays_list)
for col in df_holidays.columns:
    if df_holidays[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df_holidays[col] = df_holidays[col].apply(lambda x: json.dumps(x) if x is not None else None)

# SQL Server Bronze-a yükləyirik
df_holidays.to_sql("api_holidays", con=engine, schema="bronze", if_exists="replace", index=False)
print(f"-> {len(df_holidays)} sətir bayram məlumatı 'bronze.api_holidays' cədvəlinə yazıldı.")

# =============================================================
# 2. Frankfurter API - Valyuta Məzənnəsi (BRL -> USD)
# =============================================================
print("\n2. Valyuta kursları çəkilir (Frankfurter)...")
url_forex = "https://api.frankfurter.app/2016-01-01..2018-12-31?from=BRL&to=USD"
res_forex = requests.get(url_forex)
forex_data = res_forex.json()

# JSON olaraq save edirik
forex_json_path = api_dir / "brl_usd_exchange_rate.json"
with open(forex_json_path, "w", encoding="utf-8") as f:
    json.dump(forex_data, f, ensure_ascii=False, indent=2)

rates_dict = forex_data.get("rates", {})
df_forex = pd.DataFrame([
    {"date": date, "rate_usd": rates["USD"]} 
    for date, rates in rates_dict.items()
])

# SQL Server Bronze-a yükləyirik
df_forex.to_sql("api_exchange_rates", con=engine, schema="bronze", if_exists="replace", index=False)
print(f"-> {len(df_forex)} sətir valyuta məlumatı 'bronze.api_exchange_rates' cədvəlinə yazıldı.")

# =============================================================
# 3. Open-Meteo API - Hava Şəraiti (Weather - San Paulo)
# =============================================================
print("\n3. Hava məlumatları çəkilir (Open-Meteo)...")
lat, lon = -23.5505, -46.6333
url_weather = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={lat}&longitude={lon}&start_date=2016-01-01&end_date=2018-12-31&"
    f"daily=temperature_2m_max,temperature_2m_min,precipitation_sum&timezone=America%2FSao_Paulo"
)

res_weather = requests.get(url_weather)
weather_data = res_weather.json()

# JSON olaraq save edirik
weather_json_path = api_dir / "sao_paulo_weather.json"
with open(weather_json_path, "w", encoding="utf-8") as f:
    json.dump(weather_data, f, ensure_ascii=False, indent=2)

daily = weather_data.get("daily", {})
df_weather = pd.DataFrame({
    "date": daily.get("time", []),
    "temp_max": daily.get("temperature_2m_max", []),
    "temp_min": daily.get("temperature_2m_min", []),
    "precipitation": daily.get("precipitation_sum", [])
})

# SQL Server Bronze-a yükləyirik
df_weather.to_sql("api_weather", con=engine, schema="bronze", if_exists="replace", index=False)
print(f"-> {len(df_weather)} sətir hava məlumatı 'bronze.api_weather' cədvəlinə yazıldı.")

print("\n BÜTÜN API MƏLUMATLARI BRONZE QATINA XƏTASIZ YÜKLƏNDİ!")

1. Bayram məlumatları çəkilir (Nager.Date)...


-> 42 sətir bayram məlumatı 'bronze.api_holidays' cədvəlinə yazıldı.

2. Valyuta kursları çəkilir (Frankfurter)...
-> 768 sətir valyuta məlumatı 'bronze.api_exchange_rates' cədvəlinə yazıldı.

3. Hava məlumatları çəkilir (Open-Meteo)...
-> 1096 sətir hava məlumatı 'bronze.api_weather' cədvəlinə yazıldı.

 BÜTÜN API MƏLUMATLARI BRONZE QATINA XƏTASIZ YÜKLƏNDİ!


In [3]:
with get_engine().connect() as conn:
    print(pd.read_sql(text("""
        SELECT s.name AS schema_adi,
               t.name AS cedvel,
               SUM(p.rows) AS setir_sayi
        FROM sys.tables t
        JOIN sys.schemas s ON t.schema_id = s.schema_id
        JOIN sys.partitions p ON t.object_id = p.object_id AND p.index_id IN (0,1)
        WHERE s.name = 'bronze'
        GROUP BY s.name, t.name
        ORDER BY t.name
    """), conn))

   schema_adi                             cedvel  setir_sayi
0      bronze                 api_exchange_rates         768
1      bronze                       api_holidays          42
2      bronze                        api_weather        1096
3      bronze            olist_customers_dataset       99441
4      bronze          olist_geolocation_dataset     1000163
5      bronze          olist_order_items_dataset      112650
6      bronze       olist_order_payments_dataset      103886
7      bronze        olist_order_reviews_dataset       99224
8      bronze               olist_orders_dataset       99441
9      bronze             olist_products_dataset       32951
10     bronze              olist_sellers_dataset        3095
11     bronze  product_category_name_translation          71
